In [ ]:
import pandas as pd
from scripts.stats import friedman_test, wilcoxon_test, wilcoxon_ranking

dfs = {}
results_dir = "results_calib"

dataset = "INA_icd"
n_iter = 5

metrics = ["AUC", "F1", "Prec", "Recall", "MCC", "Acc", "Brier"]
alpha = 0.05

friedman_results = {}
wilcoxon_p_matrices = {}
wilcoxon_rankings = {}

for metric in metrics:
    df = pd.read_csv(
        f"{results_dir}/resdata_{metric}_{dataset}_ITER{n_iter}.csv",
        index_col=0
    )

    if "STATIC" in df.columns:
        df = df.drop(columns=["STATIC"])

    if "STATIC+GRU-D" in df.columns:
        df = df.drop(columns=["STATIC+GRU-D"])

    # Invert Brier so that higher is always better
    if metric == "Brier":
        df = 1 - df

    dfs[metric] = df

    stat, p = friedman_test(df, alpha=alpha, debug=True)

    friedman_results[metric] = {
        "statistic": stat,
        "p_value": p,
        "global_diff": "YES" if p < alpha else "NO"
    }

    if p < alpha:
        p_matrix, reject, p_corr, results = wilcoxon_test(
            df,
            alpha=alpha,
            debug=True
        )

        ranking = wilcoxon_ranking(
            df,
            reject,
            higher_is_better=True
        )

        wilcoxon_p_matrices[metric] = p_matrix
        wilcoxon_rankings[metric] = ranking

friedman_df = pd.DataFrame(friedman_results).T


=== FRIEDMAN TEST ===
Statistic: 64.0629
p-value: 0.000000
➡️ SIGNIFICANT global differences

=== FRIEDMAN TEST ===
Statistic: 14.2989
p-value: 0.013818
➡️ SIGNIFICANT global differences

=== FRIEDMAN TEST ===
Statistic: 42.7906
p-value: 0.000000
➡️ SIGNIFICANT global differences

=== FRIEDMAN TEST ===
Statistic: 66.2454
p-value: 0.000000
➡️ SIGNIFICANT global differences

=== FRIEDMAN TEST ===
Statistic: 12.1019
p-value: 0.033417
➡️ SIGNIFICANT global differences

=== FRIEDMAN TEST ===
Statistic: 56.6316
p-value: 0.000000
➡️ SIGNIFICANT global differences

=== FRIEDMAN TEST ===
Statistic: 4.8857
p-value: 0.429987
➡️ No global differences


/Users/maurizio/miniconda3/envs/pycaret-env/lib/python3.9/site-packages/scipy/stats/_morestats.py:4088: UserWarning: Exact p-value calculation does not work if there are zeros. Switching to normal approximation.
  warnings.warn("Exact p-value calculation does not work if there are "
/Users/maurizio/miniconda3/envs/pycaret-env/lib/python3.9/site-packages/scipy/stats/_morestats.py:4088: UserWarning: Exact p-value calculation does not work if there are zeros. Switching to normal approximation.
  warnings.warn("Exact p-value calculation does not work if there are "
/Users/maurizio/miniconda3/envs/pycaret-env/lib/python3.9/site-packages/scipy/stats/_morestats.py:4088: UserWarning: Exact p-value calculation does not work if there are zeros. Switching to normal approximation.
  warnings.warn("Exact p-value calculation does not work if there are "
/Users/maurizio/miniconda3/envs/pycaret-env/lib/python3.9/site-packages/scipy/stats/_morestats.py:4088: UserWarning: Exact p-value calculation does 

In [7]:
friedman_latex = friedman_df.copy()

friedman_latex["p_value"] = friedman_latex["p_value"].apply(
    lambda x: f"{x:.2e}"
)

friedman_latex["global_diff"] = friedman_latex["global_diff"].map({
    "YES": r"\checkmark",
    "NO": r"\ding{55}"
})

friedman_latex = friedman_latex.rename(columns={
    "statistic": "Statistic",
    "p_value": "$p$-value",
    "global_diff": "Difference"
})

friedman_latex = friedman_latex.T
latex_table = friedman_latex.to_latex(
    escape=False,
    caption="Friedman test results for all evaluation metrics.",
    label="tab:friedman",
    column_format="l" + "c"*len(friedman_latex.columns)
)

print(latex_table)

\begin{table}
\caption{Friedman test results for all evaluation metrics.}
\label{tab:friedman}
\begin{tabular}{lccccccc}
\toprule
 & AUC & F1 & Prec & Recall & MCC & Acc & Brier \\
\midrule
Statistic & 64.062857 & 14.298851 & 42.790564 & 66.245399 & 12.101947 & 56.631579 & 4.885714 \\
$p$-value & 1.75e-12 & 1.38e-02 & 4.07e-08 & 6.18e-13 & 3.34e-02 & 6.02e-11 & 4.30e-01 \\
Difference & \checkmark & \checkmark & \checkmark & \checkmark & \checkmark & \checkmark & \ding{55} \\
\bottomrule
\end{tabular}
\end{table}



In [8]:
selected_metrics = ["MCC", "Recall", "AUC"]

ranking_table = pd.DataFrame()

for metric in selected_metrics:

    ranking = wilcoxon_rankings[metric].copy()

    ranking["rank"] = ranking["score"].rank(
        method="min",
        ascending=False
    ).astype(int)

    ranking_table[metric] = ranking["rank"]

ranking_table = ranking_table[selected_metrics]

# ordinamento gerarchico:
ranking_table = ranking_table.sort_values(
    by=selected_metrics,
    ascending=True      # rank 1 migliore
)
print(ranking_table.to_latex())

\begin{tabular}{lrrr}
\toprule
 & MCC & Recall & AUC \\
\midrule
STATIC+DOME & 1 & 1 & 1 \\
STATIC+GRU & 1 & 2 & 3 \\
STATIC+M2V & 1 & 3 & 2 \\
STATIC+Dipole & 1 & 4 & 3 \\
STATIC+BiPadLSTM & 1 & 5 & 3 \\
STATIC+CEHRBERT & 1 & 6 & 3 \\
\bottomrule
\end{tabular}

